# Prompt Engineering pada Fine-Tuned Model Mistral 7B

Notebook ini digunakan untuk mengevaluasi performa prompt engineering pada model Mistral-7B-Instruct-v0.3 yang telah di-fine-tuning menggunakan metode QLoRA.

Evaluasi dilakukan pada test set menggunakan tiga strategi prompting:

1. Zero-Shot
2. One-Shot
3. Few-Shot

Hasil evaluasi kemudian dibandingkan dengan hasil prompt engineering pada base model.

In [ ]:
!nvidia-smi

## Environment Setup

In [ ]:
# Instalasi Library
!pip install -q unsloth

In [ ]:
import torch

print(torch.__version__)

from unsloth import FastLanguageModel

print("Unsloth berhasil diimport")

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

In [ ]:
import os
import re
import json

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss
)

from huggingface_hub import notebook_login
from scipy.stats import binomtest

In [ ]:
notebook_login()

In [ ]:
print("PyTorch :", torch.__version__)
print("CUDA :", torch.cuda.is_available())

## Model and Dataset Loading

In [ ]:
MODEL_NAME = "cindyzakya/mistral-dana-usability"

In [ ]:
# Load Fine-Tuned Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

In [ ]:
model.generation_config.max_length = None
model.generation_config.max_new_tokens = None

In [ ]:
test_df = pd.read_csv(
    "/kaggle/input/datasets/cindyzakya/usability-datasets-v2/test_ground_truth.csv"
)

In [ ]:
SELECTED_EXAMPLES_PATH = "/kaggle/input/datasets/cindyzakya/usability-datasets-v2/selected_prompt_examples.csv"

selected_examples_df = pd.read_csv(SELECTED_EXAMPLES_PATH)

LABELS = [
    "accuracy",
    "completeness",
    "satisfaction"
]


def load_prompt_examples(selected_examples_df):
    def select_examples(n_per_label):
        examples = []

        for label in LABELS:
            selected = (
                selected_examples_df.loc[
                    selected_examples_df["target_label"] == label
                ]
                .head(n_per_label)
            )

            if len(selected) != n_per_label:
                raise ValueError(
                    f"Jumlah contoh untuk label '{label}' harus {n_per_label}."
                )

            for _, row in selected.iterrows():
                examples.append({
                    "target_label": label,
                    "review": row["cleaned_content"],
                    "accuracy": int(row["accuracy"]),
                    "completeness": int(row["completeness"]),
                    "satisfaction": int(row["satisfaction"])
                })

        return examples

    one_shot_examples = select_examples(n_per_label=1)
    few_shot_examples = select_examples(n_per_label=2)

    return one_shot_examples, few_shot_examples


ONE_SHOT_EXAMPLES, FEW_SHOT_EXAMPLES = load_prompt_examples(selected_examples_df)

In [ ]:
print("Jumlah Data Test :", len(test_df))

test_df.head()

## Prompt Construction

In [ ]:
# Annotation guideline
def annotation_guideline():
    return """
    Anda adalah anotator usability.
    Tugas:
    Klasifikasikan setiap ulasan pengguna menggunakan multi-label classification ke dalam tiga label berikut:
    
    1. accuracy
    Label = 1 jika terdapat masalah yang menunjukkan kegagalan sistem, proses, transaksi, atau fungsi aplikasi dalam membantu pengguna mencapai tujuan penggunaan.
    Contoh:
    - error
    - bug
    - crash
    - force close
    - login gagal
    - transaksi gagal
    - OTP gagal
    - QRIS gagal
    - saldo atau dana hilang
    - pending
    - loading berkepanjangan
    - delay
    - verifikasi gagal
    - fitur tersedia tetapi tidak dapat digunakan
    - sistem menghasilkan hasil yang tidak sesuai harapan pengguna
    Fokus accuracy adalah fungsi aplikasi yang tidak berjalan sebagaimana mestinya.
    
    2. completeness
    Label = 1 jika pengguna mengeluhkan bahwa kebutuhan penggunaan belum didukung secara memadai oleh aplikasi.
    Contoh:
    - fitur tidak tersedia
    - fitur belum muncul
    - fitur kurang lengkap
    - membutuhkan fitur tambahan
    - layanan bantuan atau CS tidak membantu
    - proses upgrade akun dipersulit
    - opsi login atau akses dianggap tidak memadai
    - layanan atau fungsi yang dibutuhkan pengguna belum tersedia
    - proses penggunaan dianggap tidak lengkap atau tidak mendukung kebutuhan pengguna
    Fokus completeness adalah kelengkapan dukungan aplikasi terhadap kebutuhan pengguna.
    
    3. satisfaction
    Label = 1 jika terdapat ketidakpuasan, frustrasi, kekecewaan, kemarahan, keluhan bernada negatif, atau evaluasi negatif terhadap pengalaman penggunaan aplikasi.
    Contoh:
    - kecewa
    - kesal
    - marah
    - ribet
    - menyulitkan
    - membingungkan
    - buruk
    - parah
    - sangat mengecewakan
    Label satisfaction dapat diberikan meskipun tidak terdapat kata emosi secara eksplisit apabila isi ulasan secara keseluruhan menunjukkan pengalaman penggunaan yang negatif.
    
    Aturan penting:
    - Satu ulasan dapat memiliki lebih dari satu label.
    - Ulasan yang tidak berkaitan dengan usability dapat diberi semua label 0.
    - Label diberikan berdasarkan isi utama ulasan.
    - Fokus pada makna keseluruhan ulasan, bukan hanya keyword tertentu.
    - Keluhan teknis dapat menghasilkan accuracy = 1.
    - Keluhan teknis yang disertai nada negatif dapat menghasilkan satisfaction = 1.
    - Dana atau saldo hilang menghasilkan accuracy = 1.
    - Dana cicilan atau fitur yang belum muncul menghasilkan completeness = 1.
    - Fitur tersedia tetapi tidak dapat digunakan menghasilkan accuracy = 1.
    - CS atau layanan bantuan yang tidak membantu dapat dianggap completeness = 1 dan satisfaction = 1.
    - Jika suatu aspek tidak relevan, beri 0.
    
    Format output:
    {"accuracy":0,"completeness":0,"satisfaction":0}
    
    Output HARUS berupa JSON valid tanpa teks tambahan.
    """

In [ ]:
def example_to_assistant_json(example):
    return (
        f'{{"accuracy":{int(example["accuracy"])},'
        f'"completeness":{int(example["completeness"])},'
        f'"satisfaction":{int(example["satisfaction"])}}}'
    )

def build_zero_shot_messages(review):
    return [
        {
            "role": "system",
            "content": annotation_guideline()
        },
        {
            "role": "user",
            "content": f"Ulasan:\n{review}"
        }
    ]

def build_one_shot_messages(review):
    messages = [
        {
            "role": "system",
            "content": annotation_guideline()
        }
    ]

    for example in ONE_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": f'Ulasan:\n{example["review"]}'
        })
        messages.append({
            "role": "assistant",
            "content": example_to_assistant_json(example)
        })

    messages.append({
        "role": "user",
        "content": f"Ulasan:\n{review}"
    })

    return messages


def build_few_shot_messages(review):
    messages = [
        {
            "role": "system",
            "content": annotation_guideline()
        }
    ]

    for example in FEW_SHOT_EXAMPLES:
        messages.append({
            "role": "user",
            "content": f'Ulasan:\n{example["review"]}'
        })
        messages.append({
            "role": "assistant",
            "content": example_to_assistant_json(example)
        })

    messages.append({
        "role": "user",
        "content": f"Ulasan:\n{review}"
    })

    return messages

In [ ]:
def generate_response(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
# Prompt Experiment Runner
def run_prompt_experiment(
    df,
    prompt_builder,
    output_file,
    save_every=10
):

    if os.path.exists(output_file):

        result_df = pd.read_csv(output_file)

        start_idx = result_df["pred_accuracy"].notna().sum()

        print(f"Melanjutkan dari data ke-{start_idx}")

    else:

        result_df = df.copy()

        result_df["pred_accuracy"] = None
        result_df["pred_completeness"] = None
        result_df["pred_satisfaction"] = None

        start_idx = 0

    for idx in range(start_idx,len(result_df)):

        review = result_df.loc[
            idx,
            "cleaned_content"
        ]

        messages = prompt_builder(review)

        result = generate_response(messages)

        print(f"Processed {idx+1}/{len(result_df)}")
    
        try:
            match = re.search(
                r'\{[^{}]*\}',
                result
            )

            if not match:

                raise ValueError(
                    "JSON tidak ditemukan"
                )

            pred = json.loads(
                match.group()
            )

            pred["accuracy"] = int(pred.get("accuracy", 0))
            pred["completeness"] = int(pred.get("completeness", 0))
            pred["satisfaction"] = int(pred.get("satisfaction", 0))

            for key in [
                "accuracy",
                "completeness",
                "satisfaction"
            ]:

                if pred[key] not in [0, 1]:

                    print(f"\nWARNING DI DATA KE-{idx+1}")
                    print(f"{key}={pred[key]} tidak valid")

                    print("\nREVIEW:")
                    print(review)

                    print("\nRAW RESULT:")
                    print(result)

                    pred[key] = 0
        
        except Exception:
        
            print(f"\nERROR DI DATA KE-{idx+1}")
            print("REVIEW:")
            print(review)
        
            print("\nRAW RESULT:")
            print(result)
        
            pred = {
                "accuracy": 0,
                "completeness": 0,
                "satisfaction": 0
            }

        result_df.loc[idx,"pred_accuracy"] = pred["accuracy"]
        result_df.loc[idx,"pred_completeness"] = pred["completeness"]
        result_df.loc[idx,"pred_satisfaction"] = pred["satisfaction"]

        if (idx + 1) % save_every == 0:
            result_df.to_csv(output_file,index=False)

            print(f"Saved {idx+1}/{len(result_df)}")

    result_df.to_csv(output_file,index=False)

    print("Selesai")

    return result_df

## Inference on the Test Set

In [ ]:
# Evaluasi Zero-Shot pada Test Set
test_zero_df = run_prompt_experiment(
    df=test_df,
    prompt_builder=build_zero_shot_messages,
    output_file="finetuned_zero_shot.csv"
)

test_zero_df.head()

In [ ]:
# Evaluasi One-Shot pada Test Set
test_one_df = run_prompt_experiment(
    df=test_df,
    prompt_builder=build_one_shot_messages,
    output_file="finetuned_one_shot.csv"
)

test_one_df.head()

In [ ]:
# Evaluasi Few-Shot pada Test Set
test_few_df = run_prompt_experiment(
    df=test_df,
    prompt_builder=build_few_shot_messages,
    output_file="finetuned_few_shot.csv"
)

test_few_df.head()

## Multi-Label Evaluation

In [ ]:
def evaluate_multilabel(df):
    y_true = (
        df[LABELS]
        .fillna(0)
        .astype(int)
    )

    y_pred = (
        df[
            [
                "pred_accuracy",
                "pred_completeness",
                "pred_satisfaction"
            ]
        ]
        .fillna(0)
        .astype(int)
    )

    y_pred.columns = LABELS

    results = {
        "macro_precision": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_recall": round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "micro_precision": round(precision_score(y_true, y_pred, average="micro", zero_division=0), 4),
        "micro_recall": round(recall_score(y_true, y_pred, average="micro", zero_division=0), 4),
        "micro_f1": round(f1_score(y_true, y_pred,average="micro", zero_division=0), 4),
        "subset_accuracy": round(accuracy_score(y_true, y_pred), 4),
        "hamming_loss": round(hamming_loss(y_true, y_pred), 4)
    }

    for label in LABELS:
        results[f"{label}_precision"] = round(
            precision_score(
                y_true[label],
                y_pred[label],
                zero_division=0
            ),
            4
        )

        results[f"{label}_recall"] = round(
            recall_score(
                y_true[label],
                y_pred[label],
                zero_division=0
            ),
            4
        )

        results[f"{label}_f1"] = round(
            f1_score(
                y_true[label],
                y_pred[label],
                zero_division=0
            ),
            4
        )

    return results

In [ ]:
test_zero_df = pd.read_csv(
    "finetuned_zero_shot.csv"
)

test_one_df = pd.read_csv(
    "finetuned_one_shot.csv"
)

test_few_df = pd.read_csv(
    "finetuned_few_shot.csv"
)

In [ ]:
# Perbandingan Hasil Evaluasi pada Test Set
comparison_df = pd.DataFrame(
    [
        {
            "model": "fine-tuned",
            "dataset": "test",
            "prompting_method": "zero_shot",
            **evaluate_multilabel(test_zero_df)
        },
        {
            "model": "fine-tuned",
            "dataset": "test",
            "prompting_method": "one_shot",
            **evaluate_multilabel(test_one_df)
        },
        {
            "model": "fine-tuned",
            "dataset": "test",
            "prompting_method": "few_shot",
            **evaluate_multilabel(test_few_df)
        }
    ]
)

comparison_df

In [ ]:
comparison_df.to_csv(
    "finetuned_model_test_comparison.csv",
    index=False
)

## Comparison Before and After Fine Tuning

In [ ]:
base_comparison_df = pd.read_csv(
    "/kaggle/input/datasets/cindyzakya/output-v2/test_base_prompting_comparison.csv"
)

In [ ]:
mistral_comparison_df = pd.concat(
    [
        base_comparison_df,
        comparison_df
    ],
    ignore_index=True
)

mistral_comparison_df

In [ ]:
mistral_comparison_df.to_csv(
    "mistral_comparison.csv",
    index=False
)

In [ ]:
plt.figure(figsize=(8,5))

for model_name in mistral_comparison_df["model"].unique():

    subset = mistral_comparison_df[
        mistral_comparison_df["model"] == model_name
    ]

    plt.plot(
        subset["prompting_method"],
        subset["micro_f1"],
        marker="o",
        label=model_name
    )

plt.legend()

plt.ylabel("Micro F1")

plt.title(
    "Base Model vs Fine-Tuned Model"
)

plt.show()

In [ ]:
# Load data
base = pd.read_csv("/kaggle/input/datasets/cindyzakya/output-v2/test_base_prompting_comparison.csv")
finetuned = pd.read_csv("/kaggle/input/datasets/cindyzakya/output-v2/finetuned_model_test_comparison.csv")

metrics = {
    "subset_accuracy": "Subset Accuracy",
    "macro_precision": "Macro Precision",
    "macro_recall": "Macro Recall",
    "macro_f1": "Macro F1-score",
    "micro_precision": "Micro Precision",
    "micro_recall": "Micro Recall",
    "micro_f1": "Micro F1-score",
    "hamming_loss": "Hamming Loss"
}

strategies = base["prompting_method"]

# Visualisasi seluruh metrik
for metric, title in metrics.items():

    base_values = base[metric]
    finetuned_values = finetuned[metric]

    x = np.arange(len(strategies))
    width = 0.35

    plt.figure(figsize=(7,5))

    plt.bar(x - width/2, base_values, width, label="Base Model")
    plt.bar(x + width/2, finetuned_values, width, label="Fine-Tuned Model")

    plt.xticks(x, strategies)
    plt.xlabel("Prompting Strategy")
    plt.ylabel(title)
    plt.title(f"Comparison of {title}")

    ymin = max(0, min(base_values.min(), finetuned_values.min()) - 0.05)
    ymax = min(1.0, max(base_values.max(), finetuned_values.max()) + 0.05)
    plt.ylim(ymin, ymax)

    plt.legend()

    # Tampilkan nilai pada setiap batang
    for i, v in enumerate(base_values):
        plt.text(i - width/2, v + 0.005, f"{v:.3f}",
                 ha="center", fontsize=9)

    for i, v in enumerate(finetuned_values):
        plt.text(i + width/2, v + 0.005, f"{v:.3f}",
                 ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()

## Statistical Significance Testing

In [ ]:
base_df = pd.read_csv("/kaggle/input/datasets/cindyzakya/output-v2/test_zero_shot.csv")

print(base_df.columns.tolist())

In [ ]:
ft_df = pd.read_csv("/kaggle/input/datasets/cindyzakya/output-v2/finetuned_zero_shot.csv")

print(ft_df.columns.tolist())

In [ ]:
# Uji Statistik Inferensial: McNemar Test
BASE_PRED_LABELS = ["accuracy", "completeness", "satisfaction"]
FT_PRED_LABELS = ["pred_accuracy", "pred_completeness", "pred_satisfaction"]

BASE_DIR = "/kaggle/input/datasets/cindyzakya/output-v2"

PREDICTION_FILES = {
    "zero_shot": {
        "base": f"{BASE_DIR}/test_zero_shot.csv",
        "fine_tuned": f"{BASE_DIR}/finetuned_zero_shot.csv"
    },
    "one_shot": {
        "base": f"{BASE_DIR}/test_one_shot.csv",
        "fine_tuned": f"{BASE_DIR}/finetuned_one_shot.csv"
    },
    "few_shot": {
        "base": f"{BASE_DIR}/test_few_shot.csv",
        "fine_tuned": f"{BASE_DIR}/finetuned_few_shot.csv"
    }
}

def get_correct_predictions(base_df, ft_df):
    y_true = ft_df[LABELS].fillna(0).astype(int).reset_index(drop=True)

    base_pred = (
        base_df[BASE_PRED_LABELS]
        .fillna(0)
        .astype(int)
        .reset_index(drop=True)
    )
    base_pred.columns = LABELS

    ft_pred = (
        ft_df[FT_PRED_LABELS]
        .fillna(0)
        .astype(int)
        .reset_index(drop=True)
    )
    ft_pred.columns = LABELS

    base_correct = (y_true == base_pred).all(axis=1)
    ft_correct = (y_true == ft_pred).all(axis=1)

    return base_correct, ft_correct

def mcnemar_exact_test(base_correct, ft_correct):
    base_correct = pd.Series(base_correct).astype(bool)
    ft_correct = pd.Series(ft_correct).astype(bool)

    both_correct = ((base_correct == True) & (ft_correct == True)).sum()
    base_only_correct = ((base_correct == True) & (ft_correct == False)).sum()
    fine_tuned_only_correct = ((base_correct == False) & (ft_correct == True)).sum()
    both_wrong = ((base_correct == False) & (ft_correct == False)).sum()

    discordant_pairs = base_only_correct + fine_tuned_only_correct

    p_value = (
        1.0 if discordant_pairs == 0 else
        binomtest(
            min(base_only_correct, fine_tuned_only_correct),
            n=discordant_pairs,
            p=0.5,
            alternative="two-sided"
        ).pvalue
    )

    return {
        "both_correct": both_correct,
        "base_only_correct": base_only_correct,
        "fine_tuned_only_correct": fine_tuned_only_correct,
        "both_wrong": both_wrong,
        "discordant_pairs": discordant_pairs,
        "p_value": p_value
    }

mcnemar_results = []

for strategy, files in PREDICTION_FILES.items():
    base_df = pd.read_csv(files["base"])
    ft_df = pd.read_csv(files["fine_tuned"])

    base_correct, ft_correct = get_correct_predictions(base_df, ft_df)

    result = mcnemar_exact_test(base_correct, ft_correct)
    result["prompting_strategy"] = strategy
    result["alpha"] = 0.05
    result["significant"] = result["p_value"] < 0.05

    mcnemar_results.append(result)

mcnemar_df = pd.DataFrame(mcnemar_results)

mcnemar_df = mcnemar_df[
    [
        "prompting_strategy",
        "both_correct",
        "base_only_correct",
        "fine_tuned_only_correct",
        "both_wrong",
        "discordant_pairs",
        "p_value",
        "alpha",
        "significant"
    ]
]

mcnemar_df.to_csv("mcnemar_subset_accuracy_results.csv", index=False)

mcnemar_df